In [1]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import copy
import pandas as pd
import librosa
import shutil
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

from modules.audio import LungSound
from modules.transforms import *

In [2]:
DATA_PATH = Path(os.path.join(os.path.dirname(os.getcwd()), "data"))
RAW_DATA_FOLDER = DATA_PATH / "raw"
PREPROCESSED_DATA_FOLDER = DATA_PATH / "preprocessed"

if not os.path.exists(RAW_DATA_FOLDER):
    raise FileNotFoundError(f"Raw data folder not found at {RAW_DATA_FOLDER}. Please ensure the original data was already downloaded and placed in the correct location.")

if not os.path.exists(PREPROCESSED_DATA_FOLDER):
    os.makedirs(PREPROCESSED_DATA_FOLDER)
else:
    if len(os.listdir(PREPROCESSED_DATA_FOLDER)) > 0:
        print(f"[WARNING] Preprocessed data folder already exist and is not empty ({PREPROCESSED_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

[WARNING] Preprocessed data folder already exist and is not empty (/home/leticialopes/projects/IA901/IA901_Project/data/preprocessed). Consider deleting it to run the preprocessing step again.


In [3]:
TARGET_SR = 22050   # Hz
WINDOW_LENGTH = 5.0 # seconds
HOP_LENGTH = 5.0    # seconds

def preprocess_and_save(original_data_path: Path, preprocessed_data_path: Path):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
    """
    # Grab all files in the original data directory, .wav or not, including subdirectories
    all_files = sorted(original_data_path.glob("**/*.*"))
    # Iterate through all files and apply preprocessing to audio files, while copying non-audio files
    wav_count = 0
    for file in tqdm(all_files, desc=f"Preprocessing {original_data_path.name}"):
        if file.suffix.lower() == ".wav":
            wav_count += 1
            # Load the audio file using the LungSound class
            audio = LungSound(str(file))
            # Apply preprocessing transforms
            # 1. Split the audio into windows of fixed duration
            cropped_audios = Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH)(audio)
            for i, cropped_audio in enumerate(cropped_audios):
                # 2. Resample the audio to the target sampling rate
                resampled_audio = Resample(target_sr=TARGET_SR)(cropped_audio)
                # 3. Normalize the audio to have zero mean and unit variance
                normalized_audio = NormalizeAudio()(resampled_audio)
                # Save the preprocessed audio to the new location
                new_file_name = f"{file.stem}_crop{i}.wav"
                relative_path = file.parent.relative_to(original_data_path)
                preprocessed_file_path = preprocessed_data_path / relative_path / new_file_name
                preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(preprocessed_file_path, normalized_audio.audio, normalized_audio.sr)

            # duration = audio.duration
            # start = 0.0
            # end = start + WINDOW_LENGTH
            # while end <= duration:
            #     transformed_audio = copy.deepcopy(audio)
            #     # 1. Crop the audio to the specified window length
            #     transformed_audio = Crop(start_time=start, end_time=end)(transformed_audio)
            #     # 2. Resample the audio to the target sampling rate
            #     transformed_audio = Resample(target_sr=TARGET_SR)(transformed_audio)
            #     # 3. Normalize the audio to have zero mean and unit variance
            #     transformed_audio = NormalizeAudio()(transformed_audio)

            #     # Save the preprocessed audio to the new location
            #     new_file_name = f"{file.stem}_start{int(start)}_end{int(end)}.wav"
            #     relative_path = file.parent.relative_to(original_data_path)
            #     preprocessed_file_path = preprocessed_data_path / relative_path / new_file_name
            #     preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)
            #     sf.write(preprocessed_file_path, transformed_audio.audio, transformed_audio.sr)
            #     # Update the start and end times for the next window
            #     start += HOP_LENGTH
            #     end += HOP_LENGTH
        else:
            # If it's not an audio file, simply copy it to the new location
            relative_path = file.parent.relative_to(original_data_path)
            new_file_path = preprocessed_data_path / relative_path / file.name
            new_file_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(file, new_file_path)

    # Save a file containing the total number of preprocessed audio files
    with open(preprocessed_data_path / "preprocessing.txt", "w") as f:
        f.write(f"Total preprocessed audio files: {wav_count}\n")
        f.write("Preprocessing steps applied:\n")
        f.write(f"1. Crop the audio to {int(WINDOW_LENGTH)}-second windows with a hop length of {int(HOP_LENGTH)} seconds.\n")
        f.write(f"2. Resample the audio to a target sampling rate of {TARGET_SR} Hz.\n")
        f.write(f"3. Normalize the audio to have zero mean and unit variance.\n")

In [4]:
preprocess_and_save(RAW_DATA_FOLDER, PREPROCESSED_DATA_FOLDER)

Preprocessing raw:   0%|          | 0/2194 [00:00<?, ?it/s]